# Основные понятия и примеры прикладных задач

# Используемые пакеты

In [ ]:
# При отсутствии библиотек выполните в отдельной ячейке:
# %pip install numpy pandas scikit-learn matplotlib seaborn


In [ ]:
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Ирисы Фишера
Выборка взята отсюда: https://archive.ics.uci.edu/ml/datasets/iris

## Загрузка выборки

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
dataset = pd.DataFrame(iris.data, columns=['длина чашелистика', 'ширина чашелистика', 'длина лепестка', 'ширина лепестка'])
dataset['класс'] = ['Iris-' + iris.target_names[i] for i in iris.target]
dataset.sample(5, random_state=0)

## Начало работы с данными
1. Определить множество объектов:
    * Определить размер выборки
    * Определить признаки, которыми описываются объекты
2. Определить множество ответов
3. Определить тип задачи машинного обучения
6. ...

### Множество объектов
В данной задачи множество объектов описывается $n=4$ признаками:
1. Длина чашелистика
2. Ширина чашелистика
3. Длина лепестка
4. Ширина лепестка

In [ ]:
print('Размер выборки составляет l={} объектов.'.format(len(dataset)))

Все признаки являются вещественными признаками. Формально объекты $\mathbf{X}$ представляються в следующем виде:
$$\mathbf{X} \in \mathbb{R}^{l\times n},$$
где $l$ число объектов, а $n$ число признаков.

Получаем, что $\mathbf{X}$ это некоторая вещественная матрица размера $l\times n$.

### Множество ответов
В данной задаче множество ответов состоит из трех элементов:
1. Iris-virginica
2. Iris-versicolor
3. Iris-setosa

### Задача машинного обучения
В нашем случае, так как мощность множества $|\mathbf{y}|=3 \ll l=150$ получаем задачу классификации на $M=3$ класса.

## Анализ данных
Сначала проэктируем все объекты на двумерные плоскости, для упрощения анализа

In [ ]:
sns.pairplot(dataset, hue='класс', height=2)
plt.show()

Из рисунка видно, что класс синих точек (Iris-setosa) легко отделяется от двух других цветов. Оранджевые и зеленные точки отделяются не так просто в каждой из проэкций, но все равно можно провести прямую, которая отделит оранджевые точки от зеленых.

## Построение модели
### Преобразование данных
Как было сказано ранее нам требуется решить задачу классификации на 3 класса. Но для наглядноси рассмотрим бинарную классификацию (классификацию на несколько классов рассмотрим в следующей лекции).

Чтобы исходную задачу преобразовать в задачу бинарной классификации уберем из выборки все объекта класса Iris-setosa.

In [ ]:
binary_dataset = dataset.drop(index=dataset.index[dataset['класс'] == 'Iris-setosa'])

Классы закодируем целыми числами $-1$ и $1$.

In [ ]:
binary_dataset = binary_dataset.copy()
binary_dataset['класс'] = binary_dataset['класс'].map({'Iris-versicolor': -1, 'Iris-virginica': 1})

Получаем задачу бинарной классификации.

### Модель алгоритмов

Модель алгоритмов $\mathfrak{F}$ в машинном обучении это некоторое множество функций, которые действуют из множества объектов в множество ответов, в нашем случае:
$$\mathfrak{F} = \{f| f: \mathbb{R}^n \to \{-1, 1\}, \text{еще какие-то ограничения}\},$$
обычно $\mathfrak{F}$ это некоторое параметрическое семество функций, тоесть разные функции $f$ отличаются друг от друга только каким-то параметром. Простым примером параметрическим семейством функций для задачи бинарной классификации является семейство линейный классификатор:
$$\mathfrak{F}_{bcl} = \left\{f\bigr(\theta, \mathbf{x}\bigr)=\text{sign}\bigr(\theta^{\mathsf{T}}\mathbf{x}\bigr)\bigr| \theta \in \mathbb{R}^{n} \right\}.$$


### Функция потерь

Машиное обучение это всегда выбор функции из множества $\mathfrak{F}$. Чтобы выбрать функцию, нужен некоторый критерий по которому она выбирается, то есть нужно упоррядочить все функции от худшей к лучшей. Для этого построем функционал $\mathcal{L}$, который каждой функции $f \in \mathfrak{F}$ ставит в соответствии число из $\mathbb{R}_+$. В машинном обучении обычно функционал качества водиться как некоторая ошибка на выборке. В общем виде функционал качества можно представить в следующем виде:
$$\mathcal{L}\bigr(f, \mathbf{X}, \mathbf{y}\bigr) = \sum_{i=1}^l\mathcal{q}\bigr(f, \mathbf{x}_i, y_i\bigr),$$
где $q$ некоторая функция ошибки на некотором объекте $\mathbf{x}$. Функционал качества $\mathcal{L}$ называется эмперическим риском.

### Оптимизационная задача

Далее нужно поставить задачу оптимизации для выбора $f \in \mathfrak{F}$. Здесь все просто, просто минимизируем эмперический риск:
$$\hat{f} = \arg \min_{f \in \mathfrak{F}} \mathcal{L}\bigr(f, \mathbf{X}, \mathbf{y}\bigr).$$

Важно! В результе функция $\hat{f}$ зависит от выборки $\left(\mathbf{X}, \mathbf{y}\right)$, то есть для разных наборов данных оптимальная функция будет различная.

Вернемся к нашей задаче. В нашем случае функционал качества будет иметь следующий вид:
$$\mathcal{L}\bigr(\theta, \mathbf{X}, \mathbf{y}\bigr) = \sum_{i=1}^l\bigr[f\bigr(\theta, \mathbf{x}_i\bigr) \not= y_i\bigr],$$
и оптимизационная задача переписывается в виде:
$$\hat{\theta} = \arg \min_{\theta \in \mathbb{R}^n} \sum_{i=1}^l\bigr[f\bigr(\theta, \mathbf{x}_i\bigr) \not= y_i\bigr].$$

И на самом деле в дальнейшем будем решать иммено такие задачи, на поиск оптимального параметра. Само решение задачи линейной бинарной классификации будет на следующей лекции. Сейчас используемся библиотеками для решения данной задачи. Далее в примере будет найден параметр $\hat{\theta}$ не как решение непосредственно этой оптимизационной задачи, а немного измененной, но об этом позже в следующей лекции.

### Поиск оптимального вектора параметров
Перейдем к двум матрицам:
1. Матрице объектов $\mathbf{X} \in \mathbb{R}^{l\times (n+1)}$
2. Вектору ответов $\mathbf{y} \in \{-1,1\}^l$

Заметим, что объекты мы погрузили в пространство более большой размерности, добавив еще один признак, который у всех объектов будет равен $1$.

In [ ]:
X = binary_dataset.iloc[:, binary_dataset.columns != 'класс'].values
y = binary_dataset.iloc[:, binary_dataset.columns == 'класс'].values.reshape(-1)
X = np.array(np.hstack([X, np.ones([len(X), 1])]), dtype=np.float64)
y = np.array(y, dtype=np.int64)

In [ ]:
model = LogisticRegression(random_state=0, max_iter=2000, fit_intercept=False)
_ = model.fit(X, y)

Получаем вектор оптимальных параметров $\hat{\theta}$

In [ ]:
model.coef_

Загрузите файл winequality-red.csv (с помощью pd.read_csv).

Последний столбец отвечает за качество вина. Будем считать, что хорошее вино начинается с цифры 7 и выше. 
Остальное вино будем счиатать некачественным.

Сделайте классификацию вина по данному датасету. Два класса "Хорошее" и "Не качественное". 
Результатом будем считать уравнение гиперповерхности на гиперпространстве параметров, отделяющее один класс от другого. 

# Решение: классификация качества красного вина

Код ниже самодостаточен: его можно запускать независимо от примера с ирисами. Файл `winequality-red.csv` должен находиться рядом с ноутбуком. Нужны numpy, pandas и scikit-learn.

Объект — образец вина, признаки — 11 физико-химических показателей. Цель: **1 — «Хорошее»**, если quality ≥ 7, **0 — «Не качественное»**, если quality < 7. Столбец quality исключаем из X, иначе возникнет утечка правильного ответа.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

wine = pd.read_csv(Path('winequality-red.csv'))
assert wine.shape[1] == 12 and wine.columns[-1] == 'quality'
assert not wine.isna().any().any(), 'В данных есть пропуски'
X_wine = wine.drop(columns='quality')
y_wine = (wine['quality'] >= 7).astype(int)
print('Размер данных:', wine.shape)
print('Количество пропусков:', wine.isna().sum().sum())
print('Повторяющиеся строки:', wine.duplicated().sum())
print('Не качественное:', (y_wine == 0).sum())
print('Хорошее:', (y_wine == 1).sum())
print(wine.head().to_string(index=False))


## Проверка на отложенной выборке

Используем логистическую регрессию с L2-регуляризацией (C=1). Масштабирование нужно из-за разных единиц измерения. StandardScaler обучается только на обучающей части благодаря Pipeline.

Для оценки удаляем точные повторы строк, чтобы одинаковые наблюдения не попали одновременно в обучение и тест. Делим уникальные строки в пропорции 80/20 с сохранением долей классов. Итоговую гиперплоскость далее обучаем на всех исходных данных, как требуется в задании. Две модели имеют разные назначения; тестовые метрики относятся только к первой.


In [ ]:
unique_wine = wine.drop_duplicates()
X_train, X_test, y_train, y_test = train_test_split(
    unique_wine.drop(columns='quality'),
    (unique_wine['quality'] >= 7).astype(int),
    test_size=0.2, stratify=(unique_wine['quality'] >= 7), random_state=42
)
def new_model():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=1.0, solver='lbfgs', max_iter=5000, random_state=42))

test_model = new_model()
test_model.fit(X_train, y_train)
pred = test_model.predict(X_test)
print('Обучение:', len(X_train), 'Тест:', len(X_test))
print(f'Accuracy: {accuracy_score(y_test, pred):.4f}')
print(f'Accuracy постоянного ответа 0: {(y_test == 0).mean():.4f}')
print(f'ROC AUC: {roc_auc_score(y_test, test_model.predict_proba(X_test)[:, 1]):.4f}')
print(classification_report(y_test, pred, labels=[0, 1],
    target_names=['Не качественное', 'Хорошее'], digits=4, zero_division=0))
print('Матрица ошибок: строки — истинный класс, столбцы — прогноз; порядок 0, 1')
print(confusion_matrix(y_test, pred, labels=[0, 1]))


## Итоговая модель и уравнение гиперплоскости

Вероятность хорошего вина: $p(y=1\mid x)=1/(1+e^{-g(x)})$, где $g(x)=b+\sum_{j=1}^{11}w_jx_j$.
Граница при вероятности 0,5: **$g(x)=0$**. Если $g(x)>0$, предсказываем «Хорошее», иначе «Не качественное» (при точном равенстве выбран класс 0, как в sklearn).

После стандартизации $z_j=(x_j-\mu_j)/s_j$ модель имеет коэффициенты $a_j$ и свободный член $a_0$. Возвращаемся к исходным единицам:

$$w_j=\frac{a_j}{s_j},\qquad b=a_0-\sum_{j=1}^{11}\frac{a_j\mu_j}{s_j}.$$

Свободный член учитывается через intercept_; дополнительный столбец единиц не нужен.


In [ ]:
wine_model = new_model()
wine_model.fit(X_wine, y_wine)
scaler = wine_model.named_steps['standardscaler']
lr = wine_model.named_steps['logisticregression']
w = lr.coef_[0] / scaler.scale_
b = lr.intercept_[0] - np.dot(w, scaler.mean_)
coefficients = pd.DataFrame({
    'Переменная': [f'x{i}' for i in range(1, 12)],
    'Признак': X_wine.columns,
    'Коэффициент': w
})
print(coefficients.to_string(index=False, float_format=lambda v: f'{v:.10f}'))
print(f'Свободный член b = {b:.10f}')
equation = f'{b:.10f}' + ''.join(f' {value:+.10f}*x{i}' for i, value in enumerate(w, 1)) + ' = 0'
print('Уравнение в исходных единицах:')
print(equation)

# Проверяем эквивалентность выведенной формулы и обученного Pipeline.
g = X_wine.to_numpy() @ w + b
assert np.allclose(g, wine_model.decision_function(X_wine), atol=1e-10)
assert np.array_equal((g > 0).astype(int), wine_model.predict(X_wine))
print('Формула и модель дают одинаковые прогнозы для всех строк.')


## Пример классификации

Для нового образца необходимы те же 11 признаков в исходных единицах и порядке. Ниже показан вызов на первых пяти строках; это иллюстрация применения, а не независимая оценка качества.


In [ ]:
example = wine.iloc[:5][['quality']].copy()
example['Вероятность хорошего'] = wine_model.predict_proba(X_wine.iloc[:5])[:, 1]
example['Прогноз'] = np.where(wine_model.predict(X_wine.iloc[:5]) == 1, 'Хорошее', 'Не качественное')
print(example.to_string(index=False))


## Вывод

Получена линейная разделяющая гиперплоскость в 11-мерном пространстве признаков. Ее численное уравнение и соответствие переменных столбцам приведены выше. Классы пересекаются, поэтому граница не гарантирует безошибочного разделения. Из-за преобладания некачественных вин одной accuracy недостаточно: дополнительно приведены precision, recall, F1 и ROC AUC. Логистическая регрессия минимизирует логистическую потерю с регуляризацией, а не непосредственно число ошибок.
